# Rotax 915 iS - Step 6: How many hours are left?

**In one sentence:** Estimates the hours until the engine must come off the aircraft - at its scheduled overhaul, or earlier if wear gets there first.

| Goes in | Comes out |
|---|---|
| The window plus 10 life-stage numbers (age, condition, ...) | Hours remaining |

Starts from what step 5 (*How worn is the engine?*) already learned.

Run the cells top to bottom with **Shift+Enter**. If this step was already trained with `run.py`, training just loads the saved model in a second; otherwise it trains here.

### 0. Setup

In [ ]:
# Finds the project folder wherever Jupyter was started, then loads everything.
import os, sys
d = os.path.abspath('.')
for _ in range(6):
    if os.path.exists(os.path.join(d, 'nb_helpers.py')): break
    if os.path.exists(os.path.join(d, 'validation_v4', 'nb_helpers.py')):
        d = os.path.join(d, 'validation_v4'); break
    d = os.path.dirname(d)
sys.path.insert(0, d)
from nb_helpers import *

ENGINE, PHASE = '915', 'rul'
setup(ENGINE, PHASE)

### 1. What "hours left" means
Most engines run to their scheduled overhaul (blue: a straight countdown).
Some wear out first (red) - those are the ones worth predicting, because a
calendar alone gets them wrong.

In [ ]:
tbo = C.tbo_of(ENGINE)
d0 = load_rows(ENGINE, ["engine_hours", "rul_hours_true"], split="train")
df = d0.sample(min(20000, len(d0)), random_state=0)
worn = df.rul_hours_true < np.clip(tbo - df.engine_hours, 0, None) - 1
plt.figure(figsize=(8, 3.8))
plt.scatter(df.engine_hours[~worn], df.rul_hours_true[~worn], s=2, alpha=0.4, color=HEALTHY,
            label="reaches scheduled overhaul")
plt.scatter(df.engine_hours[worn], df.rul_hours_true[worn], s=2, alpha=0.5, color=FAULTY,
            label="wears out first")
plt.xlabel("engine hours"); plt.ylabel("hours left"); plt.legend(markerscale=5)
plt.title(f"Overhaul is due at {tbo:.0f} h"); plt.tight_layout(); plt.show()
print(f"{worn.mean():.0%} of rows belong to an engine that wears out before its overhaul.")

### 2. Train the model (or load it)

In [ ]:
payload = train_or_load(ENGINE, PHASE)
print(f"Done. Trained for {payload.get('epochs_run', '?')} passes over the data.")

### 3. Scorecard
Measured on the test flights. The quality bar catches a model that did not learn - it is a safety check, not a target.

In [ ]:
scorecard(ENGINE, PHASE)

### 4. See it working on flights it never saw
True hours left against the model's estimate. Dots on the dashed line are exact.

In [ ]:
tbo = C.tbo_of(ENGINE)
r = predict_test(ENGINE, PHASE)
true, pred = r["true"].ravel(), r["pred"].ravel()
cal = calendar_rul(ENGINE, r["aux"])
worn = true < cal - 1
plt.figure(figsize=(5.5, 5.5))
plt.scatter(true[~worn], pred[~worn], s=3, alpha=0.25, color=HEALTHY, label="reaches overhaul")
plt.scatter(true[worn], pred[worn], s=4, alpha=0.5, color=FAULTY, label="wears out first")
plt.plot([0, tbo], [0, tbo], "k--", lw=0.8)
plt.xlabel("true hours left"); plt.ylabel("predicted hours left"); plt.legend(markerscale=4)
plt.title("Each dot is one 128-second window"); plt.tight_layout(); plt.show()

### 5. What this means

In [ ]:
e_model, e_clock = np.abs(pred - true), np.abs(cal - true)
print(f"Average error: {e_model.mean():.0f} h ({100 * e_model.mean() / tbo:.1f}% of the {tbo:.0f} h life).")
print(f"On engines that wear out early, the model is off by {e_model[worn].mean():.0f} h. "
      f"Simply counting down to the overhaul date would be off by {e_clock[worn].mean():.0f} h on those same engines.")

---
### Terms used

| Term | Meaning |
|---|---|
| **Window** | 128 seconds of flight data - the unit every model looks at. |
| **Twin** | A perfectly healthy simulated copy of the engine, flown through the exact same flight. |
| **Residual** | Real reading minus the healthy twin's reading. Near zero means the engine behaves as it should. |
| **Train / validation / test** | Flights used to learn / to pick the best version / to score it. A flight is only ever in one of them, so the score is on flights the model never saw. |
| **TBO** | Time between overhauls - the engine's scheduled life: 2000 h, or 1200 h for the 915. |
| **Warm start** | A step begins from what an earlier step already learned instead of from nothing. |